In [2]:
pip install igraph


In [5]:
from tqdm.notebook import tqdm
import re
import os
import shutil
import numpy as np
import pandas as pd
import igraph as ig
from scipy.sparse import lil_matrix, save_npz
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
#%load_ext autoreload
#%autoreload 2

#data_path = '../datasets/data/' # updated path
#save_path = data_path+'kg/'

# note, run pip install urllib3==1.26.6 to avoid an OpenSSL dependency in the current version of urllib3

In [6]:

def assert_dtypes(df):
    all_string = True
    for i, x in enumerate(df.dtypes.values):
        if x != np.dtype('O'):
            all_string = False
            print(df.columns[i], x)
    if not all_string: assert False

In [7]:

def clean_edges(df):
    df = df.get(['relation', 'display_relation', 'x_id','x_type', 'x_name', 'x_source','y_id','y_type', 'y_name', 'y_source'])
    df = df.dropna()
    df = df.drop_duplicates()
    df = df.query('not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source) and (x_name == y_name))')
    return df

In [8]:
df_mondo = pd.read_csv('mondo.csv')


In [9]:
df_mondo_par = pd.read_csv('mondo_parent.csv')


In [10]:
df_protein = pd.read_csv('protein.csv')


In [11]:
df_dis = pd.read_csv('disease.csv')

/tmp/ipykernel_2152/3101094912.py:1: DtypeWarning: Columns (6,9,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dis = pd.read_csv('disease.csv')


In [12]:
df_mondo

,mondoid,name,def,comment,id
0,MONDO:0000001,disease or disorder,A disease is a disposition to undergo patholog...,NaN,1
1,MONDO:0000002,"obsolete 46,XX sex reversal",NaN,NaN,2
2,MONDO:0000003,obsolete 17-hydroxysteroid dehydrogenase defic...,NaN,NaN,3
3,MONDO:0000004,adrenocortical insufficiency,An endocrine or hormonal disorder that occurs ...,NaN,4
4,MONDO:0000005,"alopecia, isolated",NaN,NaN,5
...,...,...,...,...,...
26578,MONDO:8000030,obsolete morphological anomaly,NaN,NaN,26579
26579,MONDO:8000031,obsolete subtype of a disorder,NaN,NaN,26580
26580,MONDO:8000032,obsolete malformation syndrome,NaN,NaN,26581
26581,MONDO:8000033,obsolete group of disorders,NaN,NaN,26582


In [13]:
print(df_mondo_par.columns)

Index(['mondoid', 'parentid', 'id'], dtype='object')


In [14]:
# Merging df_mondo with df_mondo_par on 'mondoid' to get parent relationships
df_dis_dis1 = pd.merge(df_mondo, df_mondo_par, how='left', on='mondoid')

# Renaming columns after the first merge
df_dis_dis1 = df_dis_dis1.rename(columns={'parentid': 'x_id', 'name': 'x_name'})

# Merging again with df_mondo on 'x_id' to get parent node names
df_dis_dis1 = pd.merge(df_dis_dis1, df_mondo, how='left', left_on='x_id', right_on='mondoid')

# Renaming columns after the second merge
df_dis_dis1 = df_dis_dis1.rename(columns={'mondoid': 'y_id', 'name': 'y_name'})

# Adding additional columns
df_dis_dis1['x_type'] = 'disease'
df_dis_dis1['x_source'] = 'MONDO'
df_dis_dis1['y_type'] = 'disease'
df_dis_dis1['y_source'] = 'MONDO'
df_dis_dis1['relation'] = 'disease_disease'
df_dis_dis1['display_relation'] = 'parent-child'

# Cleaning edges using the modified clean_edges function
#df_dis_dis1 = clean_edges(df_dis_dis1)

# Displaying the first row of the result
df_dis_dis1.head(1)


,mondoid_x,x_name,def_x,comment_x,id_x,x_id,id_y,mondoid_y,y_name,def_y,comment_y,id,x_type,x_source,y_type,y_source,relation,display_relation
0,MONDO:0000001,disease or disorder,A disease is a disposition to undergo patholog...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,disease,MONDO,disease,MONDO,disease_disease,parent-child


In [15]:
df_dis

,id,dtype,protein_id,nhprotein_id,name,did,evidence,zscore,conf,description,reference,drug_name,log2foldchange,pvalue,score,source,O2S,S2O,mondoid
0,1,UniProt Disease,1,NaN,Cystathioninuria,MIM:219500,5 8,NaN,NaN,Autosomal recessive phenotype characterized by...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0009058
1,2,UniProt Disease,6,NaN,Pilarowski-Bjornsson syndrome,MIM:617682,21,NaN,NaN,An autosomal dominant disorder characterized b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0060568
2,3,UniProt Disease,13,NaN,"Microcephaly 22, primary, autosomal recessive",MIM:617984,9,NaN,NaN,"A form of microcephaly, a disease defined as a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0054805
3,4,UniProt Disease,17,NaN,"Cone dystrophy, retinal 3A",MIM:610024,2,NaN,NaN,A rare form of cone dystrophy associated with ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0012398
4,5,UniProt Disease,24,NaN,Retinitis pigmentosa 57,MIM:613582,3,NaN,NaN,A retinal dystrophy belonging to the group of ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0013315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363129,1014718,DrugCentral Indication,899,NaN,Diabetes mellitus type 2,DOID:9352,NaN,NaN,NaN,NaN,NaN,tirzepatide,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0005148
363130,1014719,DrugCentral Indication,6345,NaN,Growth hormone deficiency,NaN,NaN,NaN,NaN,NaN,NaN,somatrogon,NaN,NaN,NaN,NaN,NaN,NaN,NaN
363131,1014720,DrugCentral Indication,17480,NaN,Prevention of cerebrovascular spasm,NaN,NaN,NaN,NaN,NaN,NaN,clazosentan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
363132,1014721,DrugCentral Indication,18311,NaN,Refractory chronic cough,NaN,NaN,NaN,NaN,NaN,NaN,gefapixant,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
import pandas as pd

column_names = df_dis.columns
column_names


Index(['id', 'dtype', 'protein_id', 'nhprotein_id', 'name', 'did', 'evidence',
       'zscore', 'conf', 'description', 'reference', 'drug_name',
       'log2foldchange', 'pvalue', 'score', 'source', 'O2S', 'S2O', 'mondoid'],
      dtype='object')

In [17]:
df_prot_dis1 = df_dis.query('dtype=="Uniprot Disease"')



df_prot_dis1 = pd.merge(df_prot_dis1, df_mondo, 'inner', left_on='did', right_on='mondoid')


In [18]:
df_prot_dis1

,id_x,dtype,protein_id,nhprotein_id,name_x,did,evidence,zscore,conf,description,...,score,source,O2S,S2O,mondoid_x,mondoid_y,name_y,def,comment,id_y


In [19]:
df_mondo

,mondoid,name,def,comment,id
0,MONDO:0000001,disease or disorder,A disease is a disposition to undergo patholog...,NaN,1
1,MONDO:0000002,"obsolete 46,XX sex reversal",NaN,NaN,2
2,MONDO:0000003,obsolete 17-hydroxysteroid dehydrogenase defic...,NaN,NaN,3
3,MONDO:0000004,adrenocortical insufficiency,An endocrine or hormonal disorder that occurs ...,NaN,4
4,MONDO:0000005,"alopecia, isolated",NaN,NaN,5
...,...,...,...,...,...
26578,MONDO:8000030,obsolete morphological anomaly,NaN,NaN,26579
26579,MONDO:8000031,obsolete subtype of a disorder,NaN,NaN,26580
26580,MONDO:8000032,obsolete malformation syndrome,NaN,NaN,26581
26581,MONDO:8000033,obsolete group of disorders,NaN,NaN,26582


In [20]:
df_prot_dis1 = df_dis.query('dtype=="UniProt Disease"')


In [21]:
df_prot_dis1

,id,dtype,protein_id,nhprotein_id,name,did,evidence,zscore,conf,description,reference,drug_name,log2foldchange,pvalue,score,source,O2S,S2O,mondoid
0,1,UniProt Disease,1,NaN,Cystathioninuria,MIM:219500,5 8,NaN,NaN,Autosomal recessive phenotype characterized by...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0009058
1,2,UniProt Disease,6,NaN,Pilarowski-Bjornsson syndrome,MIM:617682,21,NaN,NaN,An autosomal dominant disorder characterized b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0060568
2,3,UniProt Disease,13,NaN,"Microcephaly 22, primary, autosomal recessive",MIM:617984,9,NaN,NaN,"A form of microcephaly, a disease defined as a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0054805
3,4,UniProt Disease,17,NaN,"Cone dystrophy, retinal 3A",MIM:610024,2,NaN,NaN,A rare form of cone dystrophy associated with ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0012398
4,5,UniProt Disease,24,NaN,Retinitis pigmentosa 57,MIM:613582,3,NaN,NaN,A retinal dystrophy belonging to the group of ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0013315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5646,5647,UniProt Disease,20403,NaN,"Arthrogryposis, distal, 8",MIM:178110,7,NaN,NaN,"A form of distal arthrogryposis, a disease cha...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0008338
5647,5648,UniProt Disease,20404,NaN,"Deafness, autosomal dominant, 22",MIM:606346,14,NaN,NaN,A form of non-syndromic sensorineural hearing ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0011660
5648,5649,UniProt Disease,20404,NaN,"Deafness, autosomal recessive, 37",MIM:607821,15,NaN,NaN,A form of non-syndromic sensorineural hearing ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0011912
5649,5650,UniProt Disease,20404,NaN,"Deafness, autosomal dominant 22, with hypertro...",MIM:606346,16,NaN,NaN,An autosomal dominant sensorineural deafness a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MONDO:0011660


In [22]:

# Step 2: Merge with df_mondo on appropriate columns
df_prot_dis1 = pd.merge(df_prot_dis1, df_mondo, how='inner', left_on='mondoid', right_on='mondoid')
df_prot_dis1 = pd.merge(df_prot_dis1, df_mondo, how='left', left_on='mondoid', right_on='mondoid')

# Step 3: Rename columns
#df_prot_dis1 = df_prot_dis1.rename(columns={
#   'id': 'x_id',
#    'name_x': 'x_name',
#    'mondoid': 'y_id',
#    'name_y': 'y_name'
#})

# Step 4: Add new columns
df_prot_dis1['x_type'] = 'protien'
df_prot_dis1['x_source'] = 'NaN'
df_prot_dis1['y_type'] = 'disease'
df_prot_dis1['y_source'] = 'MONDO'
df_prot_dis1['relation'] = 'disease_protein'
df_prot_dis1['display_relation'] = 'associated with'

# Step 5: Clean edges (assuming clean_edges is a predefined function)
#df_prot_dis1 = clean_edges(df_prot_dis1)

# Step 6: Display the first row
df_prot_dis1


,id_x,dtype,protein_id,nhprotein_id,name_x,did,evidence,zscore,conf,description,...,name,def_y,comment_y,id,x_type,x_source,y_type,y_source,relation,display_relation
0,1,UniProt Disease,1,NaN,Cystathioninuria,MIM:219500,5 8,NaN,NaN,Autosomal recessive phenotype characterized by...,...,cystathioninuria,Cystathioninuria is an autosomal recessive dis...,NaN,9058,protien,NaN,disease,MONDO,disease_protein,associated with
1,2,UniProt Disease,6,NaN,Pilarowski-Bjornsson syndrome,MIM:617682,21,NaN,NaN,An autosomal dominant disorder characterized b...,...,Pilarowski-Bjornsson syndrome,NaN,NaN,24877,protien,NaN,disease,MONDO,disease_protein,associated with
2,3,UniProt Disease,13,NaN,"Microcephaly 22, primary, autosomal recessive",MIM:617984,9,NaN,NaN,"A form of microcephaly, a disease defined as a...",...,"microcephaly 22, primary, autosomal recessive",NaN,NaN,24807,protien,NaN,disease,MONDO,disease_protein,associated with
3,4,UniProt Disease,17,NaN,"Cone dystrophy, retinal 3A",MIM:610024,2,NaN,NaN,A rare form of cone dystrophy associated with ...,...,retinal cone dystrophy 3A,NaN,Editor note: TODO logical defs for achromatopsias,12398,protien,NaN,disease,MONDO,disease_protein,associated with
4,5,UniProt Disease,24,NaN,Retinitis pigmentosa 57,MIM:613582,3,NaN,NaN,A retinal dystrophy belonging to the group of ...,...,retinitis pigmentosa 57,Any retinitis pigmentosa in which the cause of...,NaN,13315,protien,NaN,disease,MONDO,disease_protein,associated with
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5636,5647,UniProt Disease,20403,NaN,"Arthrogryposis, distal, 8",MIM:178110,7,NaN,NaN,"A form of distal arthrogryposis, a disease cha...",...,"contractures, pterygia, and spondylocarpotarsa...",NaN,NaN,8338,protien,NaN,disease,MONDO,disease_protein,associated with
5637,5648,UniProt Disease,20404,NaN,"Deafness, autosomal dominant, 22",MIM:606346,14,NaN,NaN,A form of non-syndromic sensorineural hearing ...,...,autosomal dominant nonsyndromic deafness 22,Any autosomal dominant nonsyndromic deafness i...,NaN,11660,protien,NaN,disease,MONDO,disease_protein,associated with
5638,5649,UniProt Disease,20404,NaN,"Deafness, autosomal recessive, 37",MIM:607821,15,NaN,NaN,A form of non-syndromic sensorineural hearing ...,...,autosomal recessive nonsyndromic deafness 37,Any autosomal recessive nonsyndromic deafness ...,NaN,11912,protien,NaN,disease,MONDO,disease_protein,associated with
5639,5650,UniProt Disease,20404,NaN,"Deafness, autosomal dominant 22, with hypertro...",MIM:606346,16,NaN,NaN,An autosomal dominant sensorineural deafness a...,...,autosomal dominant nonsyndromic deafness 22,Any autosomal dominant nonsyndromic deafness i...,NaN,11660,protien,NaN,disease,MONDO,disease_protein,associated with


In [23]:
import pandas as pd

# Assuming you have a DataFrame named 'df'
df_prot_dis1.to_csv('out_prime.csv')  # Save without index
# df.to_csv('your_filename.csv', index=True)  # Save with index (optional)



In [24]:
df_protein.columns

Index(['id', 'name', 'description', 'uniprot', 'up_version', 'geneid', 'sym',
       'family', 'chr', 'seq', 'dtoid', 'stringid', 'dtoclass'],
      dtype='object')